In [ ]:
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=x.dtype,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//3*8))
        s._ = Linear(o or i//3*8, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
RETRAIN = False
try:
    MAP_ = np.fromfile('model.i','i').tolist()
    MAP = dict(map(reversed,enumerate(MAP_)))
except FileNotFoundError as _:
    print(_)
    RETRAIN = True
    MAP = {0:0}

In [ ]:
from glob import glob
DECK = [torch.tensor([MAP.setdefault(int(i),len(MAP))for i in open(i).read().split('!')[0].split()if i.isdigit()and(RETRAIN or int(i)in MAP)],dtype=torch.short,device='cuda')for i in glob('deck/**/**.ydk',recursive=True,include_hidden=True)]
DECK = [i for i in DECK if len(i)>=40]
torch.tensor(tuple(MAP),dtype=torch.int).numpy().tofile('model.i')
try:
    if RETRAIN:
        raise FileNotFoundError
    m,o,s = torch.load('model.pk','cuda',weights_only=False)
except FileNotFoundError as _:
    print(_)
    m = 64
    m = torch.compile(Sequential(
        Embedding(len(MAP), m, scale_grad_by_freq=True),
        *(Sequential(
            Res(
                LayerNorm(m),
                At(m, 8, batch_first=True),
            ),
            Res(
                LayerNorm(m),
                Swiglu(m),
            ),
        )for _ in range(3)),
        Linear(m, len(MAP))
    ))
    o = torch.optim.Adam(m.parameters(), .0008)
    s = torch.amp.GradScaler()

In [ ]:
while True:
    L = 0
    for _ in range(10000):
        o.zero_grad()
        a = [DECK[i]for i in torch.randint(len(DECK),[16],dtype=torch.short)]
        x = torch.stack([torch.constant_pad_nd(i[torch.randperm(len(i),dtype=torch.int,device=i.device)],(0,max(map(len,a))-len(i)))for i in a]).long()
        t = functional.one_hot(x[:,1:].flip(1),len(MAP)).half().cumsum_(1).flip(1).sign_().log_()
        t[~t[:,:,1:].all(2),0] = -torch.inf
        with torch.amp.autocast('cuda',torch.float16):
            y = m(x[:,:-1])
            l = -(y.log_softmax(-1)+t).logsumexp(-1).mean()
        s.scale(l).backward()
        s.step(o)
        s.update()
        L += l.data.float()
    print(L)
    torch.save((m,o,s),'model.pk')